# Notebook 07 — Correr sistemas sobre eval set

Corre baseline RAG clasico + Agente Agentic sobre las 40 Q/A del eval set y persiste resultados a JSONL. Computa metricas custom que no requieren LLM (tool routing + fuente recall). Carga `langgraph` y `chromadb`.

## 1. Setup — paths, env, eval set

In [1]:
from __future__ import annotations

import json
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

EVAL_PATH = NOTEBOOKS_DIR / "eval_set.jsonl"
RESULTS_BASELINE = NOTEBOOKS_DIR / "results_baseline.jsonl"
RESULTS_AGENT = NOTEBOOKS_DIR / "results_agent.jsonl"

load_dotenv(PROJECT_ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en .env"

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"LLM:             {OPENAI_MODEL}")
print(f"Embeddings:      {EMBEDDING_MODEL}")
print(f"Eval set:        {EVAL_PATH.name} (existe: {EVAL_PATH.exists()})")
print(f"Results baseline: {RESULTS_BASELINE.name} (existe: {RESULTS_BASELINE.exists()})")
print(f"Results agente:   {RESULTS_AGENT.name} (existe: {RESULTS_AGENT.exists()})")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")


EVAL_SET = load_jsonl(EVAL_PATH)
print(f"\nEval set cargado: {len(EVAL_SET)} items")
print("Categorias:", dict(Counter(it["categoria"] for it in EVAL_SET)))

LLM:             gpt-4o-mini
Embeddings:      text-embedding-3-small
Eval set:        eval_set.jsonl (existe: True)
Results baseline: results_baseline.jsonl (existe: True)
Results agente:   results_agent.jsonl (existe: True)

Eval set cargado: 40 items
Categorias: {'mundial-wiki': 8, 'mundial-reglamento': 6, 'mundial-kaggle': 6, 'mundial-calendario': 6, 'plataforma': 8, 'multi-hop': 4, 'conversacional': 2}


## 2. Cargar Chroma + LLM + embedder

Mismo modelo de embeddings que F5/F6 (`text-embedding-3-small`). Critico: usar `query_embeddings` precomputados, no `query_texts`.

In [2]:
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=os.getenv("OPENAI_API_KEY"))
llm = ChatOpenAI(model=OPENAI_MODEL, api_key=os.getenv("OPENAI_API_KEY"), temperature=0)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col_mundial = client.get_collection("mundial")
col_plataforma = client.get_collection("plataforma")
print(f"Mundial:    {col_mundial.count()} chunks")
print(f"Plataforma: {col_plataforma.count()} chunks")

Mundial:    4497 chunks
Plataforma: 316 chunks


## 3. Baseline RAG clasico

Mismo codigo que NB05 §4 — un solo paso retrieve→generate con routing por keywords. Sin tool calling, sin acumulacion de mensajes.

In [3]:
def retrieve_from(collection, query, k=4):
    q_emb = embedder.embed_query(query)
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"content": d, "metadata": m, "score": 1 - dist}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


def format_context(chunks, max_chars_per_chunk=None):
    parts = []
    for c in chunks:
        src = c["metadata"].get("titulo", c["metadata"].get("ruta", "?"))
        body = c["content"]
        if max_chars_per_chunk and len(body) > max_chars_per_chunk:
            body = body[:max_chars_per_chunk].rstrip() + " [...]"
        parts.append(f"[{src}]\n{body}")
    return "\n\n".join(parts)


BASELINE_PROMPT = (
    "Eres un asistente de preguntas y respuestas en español sobre el Mundial 2026 y la plataforma 91 de predicciones. "
    "Responde SOLO con la informacion del contexto. Si no esta, di 'no lo se' — no inventes. "
    "Cita la fuente entre corchetes cuando uses informacion concreta.\n\n"
    "Pregunta: {question}\n\n"
    "Contexto:\n{context}"
)

PLATAFORMA_KEYWORDS = {
    "tribu", "tribus", "prediccion", "predicciones", "predecir", "ranking",
    "torneo", "torneos", "moneda", "monedas", "tienda", "referido", "referidos",
    "91", "noventayuno", "plataforma", "perfil", "registro", "cuenta",
}


def pick_collection(question):
    q_lower = question.lower()
    if any(kw in q_lower for kw in PLATAFORMA_KEYWORDS):
        return col_plataforma, "plataforma"
    return col_mundial, "mundial"


def rag_clasico(question, k=4):
    col, col_name = pick_collection(question)
    chunks = retrieve_from(col, question, k=k)
    context = format_context(chunks)
    prompt = BASELINE_PROMPT.format(question=question, context=context)
    answer = llm.invoke([{"role": "user", "content": prompt}]).content
    return {
        "answer": answer,
        "collection": col_name,
        "retrieved": [c["content"] for c in chunks],
        "fuentes": [c["metadata"].get("titulo", "") for c in chunks],
    }


# Smoke
r = rag_clasico("¿Cuantos equipos participan en el Mundial 2026?")
print(r["answer"])
print(f"Coleccion: {r['collection']}")
print(f"Fuentes top-{len(r['fuentes'])}: {r['fuentes']}")

En el Mundial 2026 participarán un total de 48 selecciones [Clasificación para la Copa Mundial de Fútbol de 2026].
Coleccion: mundial
Fuentes top-4: ['Clasificación para la Copa Mundial de Fútbol de 2026', 'Copa Mundial de Fútbol de 2026', 'Clasificación para la Copa Mundial de Fútbol de 2026', 'Copa Mundial de Fútbol de 2026']


## 4. Agente Agentic RAG

Mismo grafo que NB05 §9: `generate_query_or_respond → retrieve → generate_answer → END`. Sin grader/rewrite. `TOP_K_TOOL=5` (fix F6-v2).

In [4]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

TOP_K_TOOL = 5
TOOL_MAX_CHARS_PER_CHUNK = 800
RECURSION_LIMIT = 4


@tool
def buscar_mundial(query: str) -> str:
    """Busca informacion sobre el Mundial de Futbol 2026, ediciones historicas del Mundial,
    selecciones nacionales, jugadores, estadios, reglamento oficial de la FIFA, reglas del futbol,
    estadisticas y resultados historicos. Tambien cubre el CALENDARIO COMPLETO del Mundial 2026:
    los 104 partidos numerados 1-104, cada uno con fecha, hora, estadio, ciudad, fase
    (Fase de Grupos / Octavos / Cuartos / Semifinales / Tercer Puesto / FINAL = partido 104) y equipos.
    Fuentes: Wikipedia ES, Reglamento FIFA 2026 (PDF oficial), Kaggle (1930-2022) y calendario FIFA 2026.
    Usa esta tool cuando la pregunta sea sobre futbol, mundiales, FIFA, partidos especificos
    (incluyendo final, semifinales, partidos de una seleccion), jugadores, selecciones, estadios,
    fechas, horarios o reglas del juego.
    """
    chunks = retrieve_from(col_mundial, query, k=TOP_K_TOOL)
    if not chunks:
        return "Sin resultados."
    return format_context(chunks, max_chars_per_chunk=TOOL_MAX_CHARS_PER_CHUNK)


@tool
def buscar_plataforma(query: str) -> str:
    """Busca informacion sobre la plataforma 91 (noventayuno.com), una plataforma web donde los
    usuarios hacen predicciones del Mundial 2026. Cubre: como predecir partidos, sistema de puntaje,
    rankings (global, tribus, torneos), tribus (crear, unirse, owner, premios y penitencias),
    torneos (individuales, tribus, referidos), monedas internas, tienda, perfil del usuario,
    sistema de referidos, terminos y condiciones, politica de privacidad.
    Usa esta tool cuando la pregunta sea sobre como funciona la plataforma, sus features,
    o como hacer alguna accion en 91.
    """
    chunks = retrieve_from(col_plataforma, query, k=TOP_K_TOOL)
    if not chunks:
        return "Sin resultados."
    return format_context(chunks, max_chars_per_chunk=TOOL_MAX_CHARS_PER_CHUNK)


TOOLS = [buscar_mundial, buscar_plataforma]

GENERATE_PROMPT = (
    "Eres un asistente de preguntas y respuestas en español sobre el Mundial 2026 y la plataforma 91 de predicciones.\n\n"
    "REGLAS:\n"
    "1. Usa UNICAMENTE la informacion del contexto. NO inventes datos ni infieras lo que no esta escrito.\n"
    "2. Si la pregunta tiene varias partes (multi-hop) y solo encuentras informacion para algunas, responde lo que SI sabes "
    "Y declara explicitamente 'No tengo informacion sobre [parte que falta]' para el resto.\n"
    "3. Si el contexto no contiene NADA util para la pregunta, responde 'No tengo informacion sobre esto en mi base de conocimiento.'\n"
    "4. Cita la fuente entre corchetes [titulo] cuando uses informacion concreta.\n"
    "5. Se conciso (maximo 4-5 oraciones).\n"
    "6. Responde siempre en español.\n\n"
    "Pregunta: {question}\n\n"
    "Contexto recuperado:\n{context}"
)


def generate_query_or_respond(state):
    response = llm.bind_tools(TOOLS).invoke(state["messages"])
    return {"messages": [response]}


def generate_answer(state):
    messages = state["messages"]
    question = messages[0].content
    contexts = []
    for msg in reversed(messages):
        if hasattr(msg, "type") and msg.type == "tool":
            contexts.append(msg.content)
        else:
            if contexts:
                break
    contexts.reverse()
    context = "\n\n---\n\n".join(contexts) if contexts else "(sin contexto recuperado)"
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}


workflow = StateGraph(MessagesState)
workflow.add_node("generate_query_or_respond", generate_query_or_respond)
workflow.add_node("retrieve", ToolNode(TOOLS))
workflow.add_node("generate_answer", generate_answer)
workflow.add_edge(START, "generate_query_or_respond")
workflow.add_conditional_edges(
    "generate_query_or_respond", tools_condition, {"tools": "retrieve", END: END},
)
workflow.add_edge("retrieve", "generate_answer")
workflow.add_edge("generate_answer", END)
graph = workflow.compile()
print("Grafo agente compilado.")

Grafo agente compilado.


In [5]:
from langchain_core.messages import ToolMessage, AIMessage


def agente_run(question):
    """Corre el agente y extrae: respuesta final, contextos recuperados, tools llamadas."""
    state = graph.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": RECURSION_LIMIT},
    )
    messages = state["messages"]
    answer = messages[-1].content
    retrieved = [m.content for m in messages if isinstance(m, ToolMessage)]
    tools_called = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                tools_called.append(tc["name"])
    return {"answer": answer, "retrieved": retrieved, "tools_called": tools_called}


# Smoke
r = agente_run("¿En que estadio se juega la final del Mundial 2026?")
print(r["answer"])
print(f"Tools llamadas: {r['tools_called']}")
print(f"Contextos recuperados: {len(r['retrieved'])}")

La final del Mundial 2026 se disputará el domingo 19 de julio de 2026 a las 14:00 hora local (19:00 UTC) en el MetLife Stadium de East Rutherford, NJ, USA [Final del Mundial 2026]. No tengo información sobre otros estadios que puedan albergar la final.
Tools llamadas: ['buscar_mundial']
Contextos recuperados: 1


## 5. Correr baseline sobre las 40 Q/A

Persiste resultado en `results_baseline.jsonl`. Si el archivo existe, salta para no re-correr (~$0.10 por full run). Borrar el archivo manualmente para forzar re-ejecucion.

In [6]:
from tqdm.auto import tqdm


def run_baseline_all():
    results = []
    for it in tqdm(EVAL_SET, desc="baseline"):
        try:
            r = rag_clasico(it["question"], k=4)
            results.append({
                "id": it["id"],
                "categoria": it["categoria"],
                "question": it["question"],
                "ground_truth": it["ground_truth"],
                "expected_tool": it["expected_tool"],
                "fuente_esperada": it["fuente_esperada"],
                "answer": r["answer"],
                "retrieved": r["retrieved"],
                "fuentes": r["fuentes"],
                "collection": r["collection"],
                "tools_called": [],  # baseline no usa tools
            })
        except Exception as e:
            print(f"[ERR {it['id']}] {e}")
            results.append({**it, "answer": f"[ERROR] {e}", "retrieved": [], "fuentes": [], "collection": "", "tools_called": []})
    return results


if RESULTS_BASELINE.exists():
    print(f"[skip] {RESULTS_BASELINE.name} ya existe ({sum(1 for _ in open(RESULTS_BASELINE, encoding='utf-8'))} items). Borrar manualmente para re-correr.")
    baseline_results = load_jsonl(RESULTS_BASELINE)
else:
    t0 = time.time()
    baseline_results = run_baseline_all()
    save_jsonl(baseline_results, RESULTS_BASELINE)
    print(f"\nBaseline: {len(baseline_results)} resultados en {time.time() - t0:.1f}s")
    print(f"Guardado en {RESULTS_BASELINE.name}")

print("\n=== Ejemplo (primer item) ===")
ex = baseline_results[0]
print(f"Q: {ex['question']}")
print(f"A: {ex['answer'][:300]}")
print(f"Coleccion: {ex['collection']}, fuentes top-4: {ex['fuentes']}")

[skip] results_baseline.jsonl ya existe (40 items). Borrar manualmente para re-correr.

=== Ejemplo (primer item) ===
Q: ¿Cuántas selecciones participarán en la fase final del Mundial 2026?
A: Un total de 48 selecciones participarán en la fase final del Mundial 2026 [Clasificación para la Copa Mundial de Fútbol de 2026].
Coleccion: mundial, fuentes top-4: ['Clasificación para la Copa Mundial de Fútbol de 2026', 'Copa Mundial de Fútbol de 2026', 'Copa Mundial de Fútbol de 2026', 'Copa Mundial de Fútbol de 2026']


## 6. Correr agente sobre las 40 Q/A

Persiste en `results_agent.jsonl`. Tambien guarda `tools_called` para metrica de tool routing.

In [7]:
def run_agent_all():
    results = []
    for it in tqdm(EVAL_SET, desc="agente"):
        try:
            r = agente_run(it["question"])
            results.append({
                "id": it["id"],
                "categoria": it["categoria"],
                "question": it["question"],
                "ground_truth": it["ground_truth"],
                "expected_tool": it["expected_tool"],
                "fuente_esperada": it["fuente_esperada"],
                "answer": r["answer"],
                "retrieved": r["retrieved"],
                "fuentes": [],
                "collection": "",
                "tools_called": r["tools_called"],
            })
        except Exception as e:
            print(f"[ERR {it['id']}] {e}")
            results.append({**it, "answer": f"[ERROR] {e}", "retrieved": [], "fuentes": [], "collection": "", "tools_called": []})
    return results


if RESULTS_AGENT.exists():
    print(f"[skip] {RESULTS_AGENT.name} ya existe ({sum(1 for _ in open(RESULTS_AGENT, encoding='utf-8'))} items). Borrar manualmente para re-correr.")
    agent_results = load_jsonl(RESULTS_AGENT)
else:
    t0 = time.time()
    agent_results = run_agent_all()
    save_jsonl(agent_results, RESULTS_AGENT)
    print(f"\nAgente: {len(agent_results)} resultados en {time.time() - t0:.1f}s")
    print(f"Guardado en {RESULTS_AGENT.name}")

print("\n=== Ejemplo multi-hop (MH001) ===")
ex = [r for r in agent_results if r["id"] == "MH001"][0]
print(f"Q: {ex['question']}")
print(f"Tools llamadas: {ex['tools_called']}")
print(f"Retrieved: {len(ex['retrieved'])} mensajes tool")
print(f"A: {ex['answer']}")

[skip] results_agent.jsonl ya existe (40 items). Borrar manualmente para re-correr.

=== Ejemplo multi-hop (MH001) ===
Q: ¿En qué estadio juega Argentina su primer partido del Mundial 2026 y cómo puedo predecir ese partido en la plataforma 91?
Tools llamadas: ['buscar_mundial', 'buscar_plataforma']
Retrieved: 2 mensajes tool
A: Argentina juega su primer partido del Mundial 2026 contra Austria en el AT&T Stadium de Arlington, TX, USA, el lunes 22 de junio de 2026 a las 12:00 hora local [Argentina vs Austria — Fase de Grupos, Grupo J].

Para predecir ese partido en la plataforma 91, debes ingresar el marcador que crees que tendrá el partido antes de que comience. La página de predicciones está disponible en: https://www.noventayuno.com/prediccion [Cómo predecir partidos en 91].


## 7. Metricas custom — tool routing + fuente recall

**Tool routing accuracy:** ¿el agente llamo a `expected_tool`? Solo aplica al agente; el baseline no enruta por tools. Para multi-hop espera ambas tools. Para conversacional espera ninguna.

**Fuente recall:** ¿el titulo de `fuente_esperada` aparece en algun chunk de `retrieved`? Aplica a ambos sistemas. Es un proxy crudo de retrieval recall (RAGAS ContextRecall lo hace mejor con LLM-judge en NB08, pero esta metrica es rapida y reproducible).

In [8]:
def tool_match(expected, actual_list):
    if expected == "none":
        return len(actual_list) == 0
    if expected == "buscar_mundial+buscar_plataforma":
        return set(actual_list) >= {"buscar_mundial", "buscar_plataforma"}
    return expected in actual_list


def fuente_in_retrieved(fuente, retrieved_list):
    if not fuente:
        return None
    keys = [k.strip() for k in fuente.split("+")]
    blob = "\n".join(retrieved_list).lower()
    return any(k.lower() in blob for k in keys if k)


def compute_custom(results, sistema):
    tool_total, tool_ok = 0, 0
    fuente_total, fuente_ok = 0, 0
    breakdown = defaultdict(lambda: {"tool_total": 0, "tool_ok": 0, "fuente_total": 0, "fuente_ok": 0})
    for r in results:
        if sistema == "agente":
            tool_total += 1
            if tool_match(r["expected_tool"], r["tools_called"]):
                tool_ok += 1
                breakdown[r["categoria"]]["tool_ok"] += 1
            breakdown[r["categoria"]]["tool_total"] += 1
        fr = fuente_in_retrieved(r["fuente_esperada"], r["retrieved"])
        if fr is not None:
            fuente_total += 1
            breakdown[r["categoria"]]["fuente_total"] += 1
            if fr:
                fuente_ok += 1
                breakdown[r["categoria"]]["fuente_ok"] += 1
    return {
        "sistema": sistema,
        "tool_routing_accuracy": (tool_ok / tool_total) if tool_total else None,
        "fuente_recall": (fuente_ok / fuente_total) if fuente_total else None,
        "breakdown": {k: dict(v) for k, v in breakdown.items()},
    }


custom_baseline = compute_custom(baseline_results, "baseline")
custom_agent = compute_custom(agent_results, "agente")

print("=== Metricas custom ===")
print(f"BASELINE — fuente_recall: {custom_baseline['fuente_recall']:.3f}")
print(f"AGENTE   — fuente_recall: {custom_agent['fuente_recall']:.3f}")
print(f"AGENTE   — tool_routing_accuracy: {custom_agent['tool_routing_accuracy']:.3f}")

print("\nBreakdown agente por categoria:")
print(f"{'categoria':<22} {'tool_acc':>10} {'fuente':>10}")
for cat, b in custom_agent["breakdown"].items():
    tool_acc = b["tool_ok"] / b["tool_total"] if b["tool_total"] else 0
    fr = b["fuente_ok"] / b["fuente_total"] if b["fuente_total"] else float("nan")
    print(f"{cat:<22} {tool_acc:>10.3f} {fr:>10.3f}")

=== Metricas custom ===
BASELINE — fuente_recall: 0.737
AGENTE   — fuente_recall: 0.763
AGENTE   — tool_routing_accuracy: 0.925

Breakdown agente por categoria:
categoria                tool_acc     fuente
mundial-wiki                0.875      0.875
mundial-reglamento          0.833      0.500
mundial-kaggle              0.833      0.500
mundial-calendario          1.000      0.667
plataforma                  1.000      1.000
multi-hop                   1.000      1.000
conversacional              1.000        nan


## 8. Resumen NB07

In [9]:
print("=" * 60)
print("NB07 — SISTEMAS CORRIDOS")
print("=" * 60)
print(f"Eval set:    {len(EVAL_SET)} Q/A")
print(f"Sistemas:    baseline RAG clasico, agente Agentic RAG")
print(f"LLM:         {OPENAI_MODEL}")
print()
print("Outputs en notebooks/:")
print(f"  - {RESULTS_BASELINE.name}  ({len(baseline_results)} items)")
print(f"  - {RESULTS_AGENT.name}     ({len(agent_results)} items)")
print()
print("Custom:")
print(f"  fuente_recall          baseline: {custom_baseline['fuente_recall']:.3f}  agente: {custom_agent['fuente_recall']:.3f}")
print(f"  tool_routing_accuracy  agente:   {custom_agent['tool_routing_accuracy']:.3f}")
print()
print("SIGUIENTE: cerrar kernel y abrir NB08 (RAGAS) en kernel limpio.")

NB07 — SISTEMAS CORRIDOS
Eval set:    40 Q/A
Sistemas:    baseline RAG clasico, agente Agentic RAG
LLM:         gpt-4o-mini

Outputs en notebooks/:
  - results_baseline.jsonl  (40 items)
  - results_agent.jsonl     (40 items)

Custom:
  fuente_recall          baseline: 0.737  agente: 0.763
  tool_routing_accuracy  agente:   0.925

SIGUIENTE: cerrar kernel y abrir NB08 (RAGAS) en kernel limpio.
